# Tipos de Média, Distribuições e Probabilidade — Fundos de Investimento (CVM)

**Atividade:** Tipos de média · Distribuições de dados · Probabilidade e seus tipos · Conceito de modelo  
**Dataset:** Informes Diários e Cadastro de Fundos de Investimento — CVM  
**Fonte:** [dados.cvm.gov.br](https://dados.cvm.gov.br)

---

## Contexto

No notebook anterior vimos as medidas de tendência central e dispersão. Agora aprofundamos:  
- Existem **quatro tipos de média** — cada uma responde a uma pergunta diferente  
- Os dados têm **formas** (distribuições) que descrevem como os valores se organizam  
- **Probabilidade** é a linguagem matemática da incerteza — e em finanças, tudo é incerto

Aplicaremos cada conceito nos dados reais de fundos de investimento da CVM.

## 1. Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import zipfile
import io
import warnings
from scipy import stats

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 4)
sns.set_theme(style="whitegrid", palette="muted")

## 2. Download dos Dados

In [ ]:
def baixar_cvm_csv(url: str, encoding: str = "latin1") -> pd.DataFrame:
    """Baixa um arquivo CSV (ou ZIP contendo CSV) da CVM e retorna como DataFrame."""
    print(f"Baixando: {url}")
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    response = requests.get(url, timeout=120, headers=headers)
    response.raise_for_status()

    if url.endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(response.content)) as z:
            nome_csv = [f for f in z.namelist() if f.endswith(".csv")][0]
            with z.open(nome_csv) as f:
                return pd.read_csv(f, sep=";", encoding=encoding, low_memory=False)
    return pd.read_csv(io.BytesIO(response.content), sep=";", encoding=encoding, low_memory=False)


cad = baixar_cvm_csv("https://dados.cvm.gov.br/dados/FI/CAD/DADOS/cad_fi.csv")
inf = baixar_cvm_csv("https://dados.cvm.gov.br/dados/FI/DOC/INF_DIARIO/DADOS/inf_diario_fi_202502.zip")

inf["DT_COMPTC"] = pd.to_datetime(inf["DT_COMPTC"])
cad["SIT"] = cad["SIT"].str.strip().str.upper()

# Retornos diários por fundo
inf_sorted = inf.sort_values(["CNPJ_FUNDO_CLASSE", "DT_COMPTC"])
inf_sorted["retorno"] = inf_sorted.groupby("CNPJ_FUNDO_CLASSE")["VL_QUOTA"].pct_change()

print(f"\nCadastro: {cad.shape[0]:,} fundos | Informes: {inf.shape[0]:,} registros")

## 3. Tipos de Média

A palavra "média" sozinha é ambígua. Existem quatro tipos principais, cada um respondendo a uma pergunta diferente:

| Tipo | Pergunta respondida | Uso típico |
|------|--------------------|-----------|
| **Aritmética** | Qual é o valor típico somado? | Retorno esperado num único período |
| **Ponderada** | Qual é o valor típico com pesos diferentes? | Retorno de uma carteira com alocações distintas |
| **Geométrica** | Qual é a taxa de crescimento acumulado? | Retorno anualizado de um investimento |
| **Harmônica** | Qual é a média de taxas ou razões? | Preço médio de compra em aportes periódicos |

---

### 3.1 Média Aritmética

$$\bar{x} = \frac{1}{n} \sum_{i=1}^{n} x_i$$

É a soma de todos os valores dividida pelo número de observações. Simples e intuitiva, mas sensível a outliers.

### 3.2 Média Ponderada

$$\bar{x}_w = \frac{\sum_{i=1}^{n} w_i \cdot x_i}{\sum_{i=1}^{n} w_i}$$

onde $w_i$ é o peso atribuído a cada observação. Quando todos os pesos são iguais, ela se reduz à média aritmética.

**Em finanças:** o retorno de uma carteira é a média ponderada dos retornos individuais — com o patrimônio de cada fundo como peso.

### 3.3 Média Geométrica

$$\bar{x}_g = \left( \prod_{i=1}^{n} x_i \right)^{1/n} = \left( x_1 \cdot x_2 \cdots x_n \right)^{1/n}$$

Para retornos financeiros, usamos os fatores de crescimento $(1 + r_i)$:

$$\bar{x}_g = \left( \prod_{i=1}^{n} (1 + r_i) \right)^{1/n} - 1$$

**Por que importa:** a média aritmética dos retornos diários *superestima* o retorno real acumulado. A geométrica diz o que o investidor efetivamente ganhou.

### 3.4 Média Harmônica

$$\bar{x}_h = \frac{n}{\sum_{i=1}^{n} \frac{1}{x_i}}$$

Útil quando os valores são taxas ou razões. Em investimentos, é usada no **custo médio ponderado** (dollar-cost averaging): ao investir o mesmo valor todo mês, o preço médio de compra é a harmônica dos preços pagos.

In [ ]:
# Seleciona o fundo com maior número médio de cotistas (mais representativo)
fundo_ref = (
    inf.groupby("CNPJ_FUNDO_CLASSE")["NR_COTST"]
    .mean()
    .idxmax()
)
nome_ref = cad.loc[cad["CNPJ_FUNDO"] == fundo_ref, "DENOM_SOCIAL"].values
nome_ref = nome_ref[0] if len(nome_ref) > 0 else fundo_ref

ret = (
    inf_sorted[inf_sorted["CNPJ_FUNDO_CLASSE"] == fundo_ref]["retorno"]
    .dropna()
)

# Média aritmética
media_arit = ret.mean()

# Média geométrica (dos fatores de crescimento)
fatores = 1 + ret
media_geom = fatores.prod() ** (1 / len(fatores)) - 1

print(f"Fundo: {nome_ref[:60]}")
print(f"Período: {len(ret)} dias úteis\n")
print(f"Retorno diário:")
print(f"  Média aritmética : {media_arit*100:.4f}%")
print(f"  Média geométrica : {media_geom*100:.4f}%")
print(f"\nRetorno acumulado no período:")
print(f"  Via geométrica   : {(fatores.prod() - 1)*100:.4f}%")
print(f"  Via aritmética*n : {media_arit * len(ret) * 100:.4f}%  ← superestimativa")
print(f"\n→ A geométrica é a taxa real de crescimento diário que, composta, produz o retorno acumulado.")

In [ ]:
# Média ponderada: retorno da "carteira do mercado" ponderado pelo PL de cada fundo
ultimo_dia = inf["DT_COMPTC"].max()
inf_ult = inf_sorted[inf_sorted["DT_COMPTC"] == ultimo_dia].copy()
inf_ult = inf_ult.dropna(subset=["retorno", "VL_PATRIM_LIQ"])
inf_ult = inf_ult[inf_ult["VL_PATRIM_LIQ"] > 0]

media_arit_mercado   = inf_ult["retorno"].mean()
media_pond_mercado   = np.average(inf_ult["retorno"], weights=inf_ult["VL_PATRIM_LIQ"])

print(f"Data: {ultimo_dia.date()}")
print(f"Fundos com dados: {len(inf_ult):,}\n")
print(f"Retorno do mercado no dia ({ultimo_dia.date()}):")
print(f"  Média aritmética (1 fundo = 1 voto) : {media_arit_mercado*100:.4f}%")
print(f"  Média ponderada  (peso = PL)         : {media_pond_mercado*100:.4f}%")
print(f"\n→ A ponderada representa melhor o retorno de quem aloca capital proporcionalmente ao tamanho dos fundos.")

In [ ]:
# Visualização comparativa das médias
labels = ["Aritmética\n(diária)", "Geométrica\n(diária)", "Ponderada\n(último dia)"]
valores = [media_arit * 100, media_geom * 100, media_pond_mercado * 100]
cores   = ["#4C72B0", "#55A868", "#DD8452"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, valores, color=cores, edgecolor="white", width=0.4)

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.0002,
            f"{val:.4f}%", ha="center", va="bottom", fontsize=10)

ax.axhline(0, color="gray", linewidth=0.8)
ax.set_ylabel("Retorno (%)", fontsize=11)
ax.set_title("Comparação entre Tipos de Média — Dados CVM (fev/2025)", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Distribuições de Dados

Uma **distribuição** descreve como os valores de uma variável estão organizados — quais valores são comuns, quais são raros e qual é a forma geral do conjunto.

### Por que importa?

Conhecer a distribuição dos dados permite:
- Escolher a medida de tendência central correta
- Identificar outliers
- Selecionar modelos estatísticos adequados
- Calcular probabilidades

---

### 4.1 Distribuição Normal (Gaussiana)

$$f(x) = \frac{1}{\sigma\sqrt{2\pi}} \, e^{-\frac{1}{2}\left(\frac{x-\mu}{\sigma}\right)^2}$$

onde $\mu$ é a média e $\sigma$ é o desvio padrão. É simétrica em torno da média, com caudas que se estendem infinitamente.

**Propriedade da regra empírica (68-95-99,7%):**
- ~68% dos valores estão entre $\mu \pm 1\sigma$
- ~95% estão entre $\mu \pm 2\sigma$
- ~99,7% estão entre $\mu \pm 3\sigma$

**Em finanças:** retornos diários são *aproximadamente* normais — mas na prática têm **caudas mais pesadas** (maior probabilidade de eventos extremos do que a normal prevê).

### 4.2 Distribuição Lognormal

Se $\ln(X)$ segue uma distribuição normal, então $X$ segue uma **lognormal**. É assimétrica à direita e só assume valores positivos.

**Em finanças:** preços de ativos (como cotas de fundos) tendem a seguir distribuição lognormal, pois não podem ser negativos e crescem de forma multiplicativa.

### 4.3 Distribuição Uniforme

Todos os valores têm a mesma probabilidade de ocorrer. Usada como referência ou em simulações.

### 4.4 Distribuição Binomial

$$P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$

Modela o número de sucessos em $n$ tentativas independentes, cada uma com probabilidade $p$ de sucesso.

**Em finanças:** número de dias com retorno positivo em $n$ pregões.

### 4.5 Distribuição de Poisson

$$P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}$$

Modela o número de eventos raros em um intervalo de tempo, com taxa média $\lambda$.

**Em finanças:** número de defaults em uma carteira de crédito por trimestre.

In [ ]:
# Distribuição dos retornos diários vs. curva normal teórica
ret_clean = ret[(ret > ret.quantile(0.01)) & (ret < ret.quantile(0.99))]
mu, sigma = ret_clean.mean(), ret_clean.std()

x = np.linspace(ret_clean.min(), ret_clean.max(), 300)
normal_teorica = stats.norm.pdf(x, mu, sigma)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(ret_clean * 100, bins=25, density=True, color="#4C72B0",
        edgecolor="white", alpha=0.8, label="Retornos reais")
(ret_clean * 100).plot.kde(ax=ax, color="#DD8452", linewidth=2, label="KDE (forma real)")
ax.plot(x * 100, normal_teorica / 100, color="#55A868", linewidth=2,
        linestyle="--", label=f"Normal teórica (μ={mu*100:.4f}%, σ={sigma*100:.4f}%)")

ax.set_xlabel("Retorno diário (%)", fontsize=11)
ax.set_ylabel("Densidade", fontsize=11)
ax.set_title(f"Distribuição dos Retornos: Real vs. Normal Teórica\n{nome_ref[:60]}", fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Teste de normalidade (Shapiro-Wilk)
stat, p_valor = stats.shapiro(ret_clean)
print(f"\nTeste de Shapiro-Wilk (normalidade):")
print(f"  Estatística W : {stat:.4f}")
print(f"  p-valor       : {p_valor:.4f}")
if p_valor < 0.05:
    print("  Conclusão: rejeita-se a hipótese de normalidade (p < 0,05)")
else:
    print("  Conclusão: não há evidência suficiente para rejeitar normalidade")

In [ ]:
# Distribuição lognormal do valor da cota
cotas = (
    inf_sorted[inf_sorted["DT_COMPTC"] == ultimo_dia]["VL_QUOTA"]
    .dropna()
)
cotas = cotas[cotas > 0]
log_cotas = np.log(cotas)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(cotas.clip(upper=cotas.quantile(0.95)), bins=50,
             color="#4C72B0", edgecolor="white", alpha=0.85)
axes[0].set_title("Cota dos fundos (escala original)\n→ Assimétrica à direita", fontsize=11)
axes[0].set_xlabel("VL_QUOTA (R$)", fontsize=10)
axes[0].set_ylabel("Frequência", fontsize=10)

axes[1].hist(log_cotas, bins=50, color="#55A868", edgecolor="white", alpha=0.85)
x_log = np.linspace(log_cotas.min(), log_cotas.max(), 200)
pdf_log = stats.norm.pdf(x_log, log_cotas.mean(), log_cotas.std())
ax2 = axes[1].twinx()
ax2.plot(x_log, pdf_log, color="#DD8452", linewidth=2, label="Normal teórica")
ax2.set_ylabel("Densidade", fontsize=10)
ax2.legend(fontsize=9)
axes[1].set_title("ln(Cota) dos fundos (escala log)\n→ Aproxima-se da normal", fontsize=11)
axes[1].set_xlabel("ln(VL_QUOTA)", fontsize=10)
axes[1].set_ylabel("Frequência", fontsize=10)

plt.suptitle("Lognormalidade: Preços de Cotas de Fundos CVM", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print(f"Assimetria da cota original : {cotas.skew():.2f}  (positiva → cauda à direita)")
print(f"Assimetria do log da cota   : {log_cotas.skew():.2f}  (próxima de zero → aproxima normal)")

## 5. O que é Probabilidade?

**Probabilidade** é uma medida numérica entre 0 e 1 que quantifica o grau de certeza (ou incerteza) sobre a ocorrência de um evento.

$$0 \leq P(A) \leq 1$$

- $P(A) = 0$ → o evento é impossível
- $P(A) = 1$ → o evento é certo
- $P(A) = 0{,}5$ → o evento tem 50% de chance de ocorrer

### Axiomas de Kolmogorov

Toda teoria de probabilidade parte de três axiomas fundamentais:

1. **Não negatividade:** $P(A) \geq 0$ para todo evento $A$
2. **Normalização:** $P(\Omega) = 1$, onde $\Omega$ é o espaço amostral completo
3. **Aditividade:** se $A$ e $B$ são eventos mutuamente exclusivos, então $P(A \cup B) = P(A) + P(B)$

### Interpretações de Probabilidade

| Interpretação | Como define probabilidade | Exemplo em finanças |
|--------------|--------------------------|--------------------|
| **Clássica** | Razão entre casos favoráveis e totais (equiprováveis) | Probabilidade de 1 dado dar 6 = 1/6 |
| **Frequentista** | Limite da frequência relativa em muitas repetições | P(dia positivo) = proporção de dias positivos históricos |
| **Bayesiana** | Grau de crença, atualizado com novas evidências | P(recessão) revisada após novo dado de inflação |

## 6. Tipos de Probabilidade

### 6.1 Probabilidade Simples (Marginal)

$$P(A) = \frac{\text{número de resultados favoráveis a } A}{\text{número total de resultados possíveis}}$$

Considera apenas um evento, sem condicionar a outro.

### 6.2 Probabilidade Conjunta

$$P(A \cap B) = P(A \text{ e } B \text{ ocorrem simultaneamente})$$

Se $A$ e $B$ são independentes:

$$P(A \cap B) = P(A) \cdot P(B)$$

### 6.3 Probabilidade Condicional

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}, \quad P(B) > 0$$

É a probabilidade de $A$ **dado que** $B$ já ocorreu. Reduz o espaço amostral ao que é compatível com $B$.

### 6.4 Teorema de Bayes

$$P(A \mid B) = \frac{P(B \mid A) \cdot P(A)}{P(B)}$$

Permite **atualizar** crenças sobre $A$ ao observar evidência $B$. É o núcleo da inferência bayesiana.

| Termo | Nome | Significado |
|-------|------|-------------|
| $P(A)$ | Prior | Crença inicial sobre $A$ |
| $P(B \mid A)$ | Verossimilhança | Probabilidade de observar $B$ se $A$ for verdadeiro |
| $P(A \mid B)$ | Posterior | Crença atualizada sobre $A$ após observar $B$ |

In [ ]:
# Probabilidade simples: P(retorno positivo no dia)
ret_dia = inf_sorted[inf_sorted["DT_COMPTC"] == ultimo_dia]["retorno"].dropna()

p_positivo = (ret_dia > 0).mean()
p_negativo = (ret_dia < 0).mean()
p_zero     = (ret_dia == 0).mean()

print(f"Probabilidade simples — retorno dos fundos em {ultimo_dia.date()}:")
print(f"  P(retorno > 0)  = {p_positivo:.4f}  ({p_positivo*100:.1f}%)")
print(f"  P(retorno < 0)  = {p_negativo:.4f}  ({p_negativo*100:.1f}%)")
print(f"  P(retorno = 0)  = {p_zero:.4f}   ({p_zero*100:.1f}%)")
print(f"  Soma            = {p_positivo + p_negativo + p_zero:.4f}  ← deve ser 1")

In [ ]:
# Probabilidade condicional: P(retorno positivo | tipo de fundo = FIA)
# Unir com cadastro para obter tipo de fundo
inf_tipo = inf_sorted[inf_sorted["DT_COMPTC"] == ultimo_dia].merge(
    cad[["CNPJ_FUNDO", "TP_FUNDO"]],
    left_on="CNPJ_FUNDO_CLASSE",
    right_on="CNPJ_FUNDO",
    how="left"
).dropna(subset=["retorno", "TP_FUNDO"])

# P(retorno > 0 | tipo) para os 5 tipos mais comuns
tipos_comuns = inf_tipo["TP_FUNDO"].value_counts().head(5).index
inf_tipo_top = inf_tipo[inf_tipo["TP_FUNDO"].isin(tipos_comuns)]

prob_cond = (
    inf_tipo_top.groupby("TP_FUNDO")["retorno"]
    .apply(lambda x: (x > 0).mean())
    .sort_values(ascending=False)
    .rename("P(retorno > 0 | tipo)")
)

print(f"Probabilidade condicional — P(retorno > 0 | tipo de fundo):")
print(prob_cond.to_string())
print(f"\nP(retorno > 0) marginal (todos os tipos): {p_positivo:.4f}")
print("\n→ A probabilidade de retorno positivo varia conforme o tipo de fundo.")
print("  Isso mostra que 'tipo de fundo' e 'retorno positivo' NÃO são independentes.")

In [ ]:
# Probabilidade conjunta: P(retorno > 0 E patrimônio > mediana)
pl_mediana = inf_tipo["VL_PATRIM_LIQ"].median()

ret_pos   = inf_tipo["retorno"] > 0
pl_grande = inf_tipo["VL_PATRIM_LIQ"] > pl_mediana

p_ret_pos        = ret_pos.mean()
p_pl_grande      = pl_grande.mean()
p_conjunta       = (ret_pos & pl_grande).mean()
p_independencia  = p_ret_pos * p_pl_grande

print("Probabilidade conjunta:")
print(f"  P(retorno > 0)                          = {p_ret_pos:.4f}")
print(f"  P(PL > mediana)                         = {p_pl_grande:.4f}")
print(f"  P(retorno > 0 E PL > mediana)  (real)   = {p_conjunta:.4f}")
print(f"  P(A) × P(B)                  (se indep) = {p_independencia:.4f}")

if abs(p_conjunta - p_independencia) < 0.01:
    print("\n→ Os eventos são aproximadamente INDEPENDENTES.")
else:
    print("\n→ Os eventos NÃO são independentes — há associação entre tamanho e retorno positivo.")

In [ ]:
# Binomial: quantos dias de retorno positivo esperamos em fevereiro (20 pregões)?
# Estima p com frequência histórica do fundo de referência
p_hist = (ret > 0).mean()
n_pregoes = 20

k_vals = np.arange(0, n_pregoes + 1)
prob_binom = stats.binom.pmf(k_vals, n_pregoes, p_hist)

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(k_vals, prob_binom, color="#4C72B0", edgecolor="white", alpha=0.85)
ax.axvline(n_pregoes * p_hist, color="#DD8452", linewidth=2, linestyle="--",
           label=f"E[X] = n×p = {n_pregoes * p_hist:.1f} dias")
ax.set_xlabel("Número de dias com retorno positivo", fontsize=11)
ax.set_ylabel("Probabilidade", fontsize=11)
ax.set_title(
    f"Distribuição Binomial — P(k dias positivos em {n_pregoes} pregões)\n"
    f"p = {p_hist:.4f} (frequência histórica do fundo)",
    fontsize=11
)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print(f"Probabilidade histórica de dia positivo: {p_hist:.4f} ({p_hist*100:.1f}%)")
print(f"Esperança (dias positivos em {n_pregoes} pregões): {n_pregoes * p_hist:.1f}")
print(f"P(≥ 10 dias positivos): {1 - stats.binom.cdf(9, n_pregoes, p_hist):.4f}")

## 7. O que é um Modelo?

### Definição técnica

Um **modelo estatístico** é uma representação matemática da relação entre variáveis, construída a partir de suposições sobre a distribuição dos dados. Formalmente, é uma família de distribuições de probabilidade parametrizadas:

$$\mathcal{M} = \{ P_\theta : \theta \in \Theta \}$$

onde $\theta$ são os parâmetros a serem estimados e $\Theta$ é o espaço de parâmetros admissíveis.

Todo modelo tem:
1. **Hipótese de distribuição** — qual a forma dos dados (normal, Poisson, etc.)
2. **Variáveis de entrada (features)** — o que o modelo recebe como informação
3. **Variável de saída (target)** — o que o modelo tenta estimar ou prever
4. **Parâmetros** — valores ajustados para minimizar o erro
5. **Métrica de avaliação** — como medir se o modelo é bom

Exemplos em risco de mercado:

| Modelo | Suposição central | O que estima |
|--------|-----------------|--------------|
| Normal VaR | Retornos ~ Normal$(\mu, \sigma^2)$ | Perda máxima com 95% de confiança |
| GARCH(1,1) | Volatilidade varia no tempo | $\sigma_t^2$ (variância condicional) |
| LSTM | Padrões sequenciais em séries temporais | Retorno ou ES futuro |

---

### Definição pessoal

> Um modelo é uma aposta intelectual. Você escolhe quais aspectos da realidade incluir, quais ignorar e qual forma matemática usar para ligar uma coisa à outra. Toda vez que rodo um modelo, estou dizendo: *"Acredito que a realidade funciona assim — agora vou ver se os dados concordam."*
>
> O que me fascina é que modelos nunca são verdadeiros, mas podem ser úteis. O GARCH não descreve como o mundo realmente funciona — ele captura *padrão suficiente* de como a volatilidade se comporta para que eu possa tomar decisões melhores do que sem ele. Em risco de mercado, isso tem consequências reais: um modelo ruim significa capital mal alocado, posições mal dimensionadas e perdas inesperadas.
>
> Por isso modelar é uma habilidade que vai muito além do código: exige saber quando *não* confiar no modelo.

## 8. Resumo

| Conceito | Definição resumida | Aplicação em finanças |
|----------|-------------------|----------------------|
| **Média aritmética** | Soma ÷ n | Retorno esperado de um único período |
| **Média ponderada** | Soma ponderada pelos pesos | Retorno de uma carteira com alocações distintas |
| **Média geométrica** | Taxa de crescimento composta | Retorno anualizado real de um investimento |
| **Média harmônica** | Recíproco da média dos recíprocos | Custo médio em aportes periódicos |
| **Distribuição normal** | Simétrica, caudas finas | Aproximação de retornos diários |
| **Distribuição lognormal** | Assimétrica positiva, valores > 0 | Distribuição de preços de ativos |
| **Distribuição binomial** | Número de sucessos em n tentativas | Dias positivos em n pregões |
| **Probabilidade simples** | P(A) = casos favoráveis / total | P(retorno positivo no dia) |
| **Probabilidade conjunta** | P(A e B ocorrem juntos) | P(retorno positivo e PL acima da mediana) |
| **Probabilidade condicional** | P(A \| B) — dado que B ocorreu | P(retorno positivo \| tipo de fundo = FIA) |
| **Teorema de Bayes** | Atualiza crenças com novas evidências | Revisão de P(recessão) após novo dado macro |
| **Modelo** | Representação matemática da realidade | Do VaR paramétrico ao LSTM |

---

## Referências

- [CVM — Dados Abertos de Fundos de Investimento](https://dados.cvm.gov.br)
- Magalhães, M. N.; Lima, A. C. P. — *Noções de Probabilidade e Estatística* (IME-USP)
- Triola, M. F. — *Introdução à Estatística*
- Morettin, P. A.; Toloi, C. M. — *Análise de Séries Temporais*